In [7]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec

# ── 1. 路径配置 ───────────────────────────────────────────────────────────────

XLSX_PATH = Path("/Volumes/UCL/论文工作/空气污染/清华城市健康指数-健康服务.xlsx")
OUTFILE   = Path("/Users/shirley/Desktop/plots_V2/Fig3.png")
SHEET     = "earlypeak_NZ_CL"
GAM_PRED  = Path("/Users/shirley/Desktop/plots_V2/gam_pred.csv")
GAM_RUG   = Path("/Users/shirley/Desktop/plots_V2/gam_rug.csv")

W_IN = 18 / 2.54      # 加宽以容纳6列
H_IN = 15 / 2.54
DPI  = 400

# ── 2. HS 分组配置 ────────────────────────────────────────────────────────────

HS_LEVELS = ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]
COL_GROUP = {
    "Poor":         "#d7191c",
    "Moderate":     "#f47920",
    "Intermediate": "#f6c101",
    "Good":         "#abd9e9",
    "Excellent":    "#2c7bb6",
}
Y_POS    = {lvl: (i + 1) / 3 for i, lvl in enumerate(HS_LEVELS)}
UNIQUE_Y = sorted(Y_POS.values())

# ── 3. 读取数据 ───────────────────────────────────────────────────────────────

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET, usecols=[
    "city", "mort_2020", "mort_2060", "delta_mort",
    "mortratio_2020", "mortratio_2060", "delta_mortratio",
    "city_size", "GDP_group", "HS", "region_HW", "HS_type", "HS_type2"
])
df["HS_type"] = pd.Categorical(df["HS_type"], categories=HS_LEVELS, ordered=True)
df = df.dropna(subset=["HS_type"])
df["y_pos"] = df["HS_type"].map(Y_POS)

gam_pred = pd.read_csv(GAM_PRED)
gam_rug  = pd.read_csv(GAM_RUG)

# ── 4. 全局样式 ───────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.titlesize":   7,
    "axes.titleweight": "bold",
    "axes.titlepad":    3,
    "axes.labelsize":   6,
    "xtick.labelsize":  5,
    "ytick.labelsize":  5,
})

# ── 5. 分布面板函数 ───────────────────────────────────────────────────────────

def make_panel(ax, data, xvar, xlabel, xlim, show_y_labels=False):
    rng = np.random.default_rng(42)
    for lvl in HS_LEVELS:
        grp   = data[data["HS_type"] == lvl][xvar].dropna()
        grp   = grp[np.isfinite(grp)]
        y_pos = Y_POS[lvl]
        color = COL_GROUP[lvl]
        if len(grp) < 2:
            continue
        vp = ax.violinplot(grp, positions=[y_pos], vert=False,
                           widths=0.28, showextrema=False)
        for body in vp["bodies"]:
            body.set_facecolor(color); body.set_alpha(0.15); body.set_edgecolor("none")
        jitter = rng.uniform(-0.06, 0.06, size=len(grp))
        ax.scatter(grp, y_pos + jitter, color=color, s=1.2,
                   alpha=0.30, linewidths=0, zorder=2)
        q1, med, q3 = np.percentile(grp, [25, 50, 75])
        iqr = q3 - q1
        lo  = grp[grp >= q1 - 1.5*iqr].min()
        hi  = grp[grp <= q3 + 1.5*iqr].max()
        box_h, cap_h = 0.06, 0.03
        rect = mpatches.FancyBboxPatch(
            (q1, y_pos - box_h), q3 - q1, 2*box_h,
            boxstyle="square,pad=0",
            facecolor="white", edgecolor=color,
            linewidth=0.8, alpha=0.9, zorder=3)
        ax.add_patch(rect)
        ax.plot([med, med], [y_pos-box_h, y_pos+box_h],
                color=color, linewidth=1.2, zorder=4)
        ax.plot([lo, q1], [y_pos, y_pos], color=color, linewidth=0.8, zorder=3)
        ax.plot([q3, hi], [y_pos, y_pos], color=color, linewidth=0.8, zorder=3)
        for cap_x in [lo, hi]:
            ax.plot([cap_x, cap_x], [y_pos-cap_h, y_pos+cap_h],
                    color=color, linewidth=0.8, zorder=3)
    ax.set_xlim(xlim)
    ax.set_ylim(min(UNIQUE_Y)-0.2, max(UNIQUE_Y)+0.2)
    ax.set_yticks(UNIQUE_Y)
    ax.set_yticklabels(HS_LEVELS if show_y_labels else [""]*len(UNIQUE_Y), fontsize=6)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(None)
    if "ratio" in xvar:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.2f}"))
    else:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.3)
    ax.spines["left"].set_linewidth(0.3)
    ax.grid(False)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)

# ── 6. 洛伦兹曲线函数 ────────────────────────────────────────────────────────

def lorenz_curve(values):
    v = np.array(values, dtype=float)
    v = v[np.isfinite(v) & (v >= 0)]
    v = np.sort(v)
    cum_v = np.cumsum(v)
    x = np.linspace(0, 1, len(v)+1)
    y = np.concatenate([[0], cum_v / cum_v[-1]])
    return x, y

def gini(values):
    x, y = lorenz_curve(values)
    return 1 - 2*np.trapz(y, x)

def draw_lorenz_panel(ax, col_2020, col_2060, y_label, tag):
    COL_20, COL_60 = "#ca0020",  "#4db3b3"
    ax.plot([0,1],[0,1], color="grey", linewidth=0.8, linestyle="--", zorder=1)
    for col, color, year in [(col_2020, COL_20, "2020"), (col_2060, COL_60, "2060")]:
        v = df[col].dropna().values
        x, y = lorenz_curve(v)
        g = gini(v)
        ax.fill_between(x, x, y, color=color, alpha=0.08)
        ax.plot(x, y, color=color, linewidth=1.2, label=f"{year}  Gini={g:.3f}", zorder=3)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Cumulative share of cities,\nranked by HS", fontsize=6, labelpad=2)
    ax.set_ylabel("")
    ax.text(-0.28, 0.5, y_label, transform=ax.transAxes, fontsize=6,
            rotation=90, va="center", ha="center")
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.25))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.25))
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(0.3)
    ax.grid(linewidth=0.2, color="#e0e0e0", zorder=0)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)
    ax.legend(fontsize=5, frameon=False, loc="upper left",
              handlelength=1.2, handletextpad=0.4, labelspacing=0.3)
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=7, fontweight="bold", va="bottom", ha="left")

# ── 7. GAM 样条面板函数 ───────────────────────────────────────────────────────

COLOR_FULL = "#ca0020"
COLOR_SUB  = "#4db3b3"

def fmt_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

def draw_spline_panel(ax, gam_pred, gam_rug, yvar, ylabel, tag, show_y_label=True):
    pf = gam_pred[(gam_pred["yvar"] == yvar) & (gam_pred["sample"] == "full")]
    ps = gam_pred[(gam_pred["yvar"] == yvar) & (gam_pred["sample"] == "sub")]
    edf_full = pf["edf"].iloc[0]; p_full = pf["p_val"].iloc[0]
    edf_sub  = ps["edf"].iloc[0]; p_sub  = ps["p_val"].iloc[0]
    ax.axhline(0, linestyle="--", color="grey", linewidth=0.8, zorder=1)
    ax.fill_between(pf["HS"], pf["lo"], pf["hi"], color=COLOR_FULL, alpha=0.08, linewidth=0)
    ax.fill_between(ps["HS"], ps["lo"], ps["hi"], color=COLOR_SUB,  alpha=0.30, linewidth=0)
    ax.plot(pf["HS"], pf["fit"], color=COLOR_FULL, linewidth=1.0, label="All cities")
    ax.plot(ps["HS"], ps["fit"], color=COLOR_SUB,  linewidth=1.0, label="HS ≤ 90")
    ax.autoscale(False)
    ymin, ymax = ax.get_ylim()
    rug_y = ymin - 0.02*(ymax - ymin)
    ax.plot(gam_rug["HS"].values, np.full(len(gam_rug), rug_y),
            linestyle="none", marker="|", color="black",
            alpha=0.6, markersize=3, markeredgewidth=0.4, clip_on=False)
    all_hi = np.concatenate([pf["hi"].values, ps["hi"].values])
    x_anno = gam_pred["HS"].min() + 0.02*(gam_pred["HS"].max() - gam_pred["HS"].min())
    y_anno = np.nanmax(all_hi) * 0.95
    ax.text(x_anno, y_anno,
            f"All cities: edf = {edf_full}, {fmt_p(p_full)}\n"
            f"HS ≤ 90: edf = {edf_sub}, {fmt_p(p_sub)}",
            ha="left", va="top", fontsize=5, fontfamily="Arial")
    ax.set_xlabel("Health capacity (HS)", fontsize=6, labelpad=3)
    ax.set_ylabel(ylabel if show_y_label else "", fontsize=6, labelpad=0)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.3)
    ax.spines["left"].set_linewidth(0.3)
    ax.text(-0.08, 1.03, tag, transform=ax.transAxes,
            fontsize=7, fontweight="bold", va="bottom", ha="left")

# ── 8. 图形拼合 ───────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(W_IN, H_IN), dpi=DPI)

gs = GridSpec(
    nrows=3, ncols=4,
    figure=fig,
    width_ratios=[1, 1, 1, 2],
    height_ratios=[1, 1, 1.5],
    hspace=0.45,
    wspace=0.28,
)

# Row 0
ax_mort20  = fig.add_subplot(gs[0, 0])
ax_mort60  = fig.add_subplot(gs[0, 1])
ax_dmort   = fig.add_subplot(gs[0, 2])
ax_lorenz0 = fig.add_subplot(gs[0, 3])

# Row 1
ax_mr20    = fig.add_subplot(gs[1, 0])
ax_mr60    = fig.add_subplot(gs[1, 1])
ax_dmr     = fig.add_subplot(gs[1, 2])
ax_lorenz1 = fig.add_subplot(gs[1, 3])

# Row 2
ax_si = fig.add_subplot(gs[2, 0:2])
ax_sj = fig.add_subplot(gs[2, 2:4])

# ── 绘制分布面板
make_panel(ax_mort20, df, "mort_2020",       "Premature deaths in 2020", (0, 35000),       True)
make_panel(ax_mort60, df, "mort_2060",       "Premature deaths in 2060", (0, 35000),       False)
make_panel(ax_dmort,  df, "delta_mort",      "Change in deaths",         (-20000, 20000),  False)

make_panel(ax_mr20,   df, "mortratio_2020",  "Mortality rate in 2020 (%)", (0, 0.1),       True)
make_panel(ax_mr60,   df, "mortratio_2060",  "Mortality rate in 2060 (%)", (0, 0.1),       False)
make_panel(ax_dmr,    df, "delta_mortratio", "Change in rate (%)",         (-0.05, 0.05),  False)

# ── 绘制洛伦兹曲线
draw_lorenz_panel(ax_lorenz0, "mort_2020",      "mort_2060",
                  "Cumulative share of deaths",        "b")
draw_lorenz_panel(ax_lorenz1, "mortratio_2020", "mortratio_2060",
                  "Cumulative share of mortality rate", "d")

# ── 绘制样条
draw_spline_panel(ax_si, gam_pred, gam_rug, "delta_mort",
                  "Change in deaths",       "e", show_y_label=True)
draw_spline_panel(ax_sj, gam_pred, gam_rug, "delta_mortratio",
                  "Change in mortality rate", "f", show_y_label=True)

# ── 组标签
ax_mort20.text(-0.20, 1.03, "a", transform=ax_mort20.transAxes,
               fontsize=7, fontweight="bold", va="bottom", ha="left")
ax_mr20.text(  -0.20, 1.03, "c", transform=ax_mr20.transAxes,
               fontsize=7, fontweight="bold", va="bottom", ha="left")

# ── 样条共享图例
handles, labels = ax_si.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2,
           fontsize=6, frameon=False, bbox_to_anchor=(0.5, 0.02))

# ── 9. 高度对齐 + 第三行严格 1:1 ─────────────────────────────────────────────

fig.canvas.draw()

def _match_height_to_ref(row_axes, ref_ax):
    ref_pos = ref_ax.get_position()
    ref_cy  = ref_pos.y0 + ref_pos.height / 2
    ref_h   = ref_pos.height
    for ax in row_axes:
        pos = ax.get_position()
        ax.set_position([pos.x0, ref_cy - ref_h/2, pos.width, ref_h])

_match_height_to_ref([ax_mort20, ax_mort60, ax_dmort], ax_lorenz0)
_match_height_to_ref([ax_mr20,   ax_mr60,   ax_dmr],   ax_lorenz1)

# 强制第三行严格 1:1
fig.canvas.draw()

pos_si     = ax_si.get_position()
pos_sj     = ax_sj.get_position()
row2_x0    = pos_si.x0
row2_x1    = pos_sj.x0 + pos_sj.width
row2_total = row2_x1 - row2_x0
gap        =( pos_sj.x0 - (pos_si.x0 + pos_si.width))*1.5
panel_w    = (row2_total - gap) / 2

ax_si.set_position([row2_x0,                   pos_si.y0, panel_w, pos_si.height])
ax_sj.set_position([row2_x0 + panel_w + gap,   pos_sj.y0, panel_w, pos_sj.height])

# ── 10. 保存 ──────────────────────────────────────────────────────────────────

OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"✓ Saved → {OUTFILE}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_51007/2802481749.py:126: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return 1 - 2*np.trapz(y, x)


✓ Saved → /Users/shirley/Desktop/plots_V2/Fig3.png
